# 1. Phase de groupes (4 équipes, matchs simples)

**Problème.** Quatre équipes A, B, C et D. Chaque équipe rencontre une seule fois chacune des autres, sur 3 journées. On veut lister tous les calendriers possibles.

**Variable.** $x_{ijk} = 1$ si l'équipe $i$ joue contre l'équipe $j$ lors de la journée $k$, 0 sinon. Il n'y a pas de notion de domicile, donc $x_{ijk} = x_{jik}$.

**Contraintes.**

$$x_{ijk} = x_{jik} \quad \forall i \neq j,\ \forall k \qquad \text{(symétrie)}$$

$$\sum_{k=1}^{3} x_{ijk} = 1 \quad \forall i \neq j \qquad \text{(chaque paire se rencontre une fois)}$$

$$\sum_{j \neq i} x_{ijk} = 1 \quad \forall i,\ \forall k \qquad \text{(un match par équipe et par journée)}$$

Pas de fonction objectif : on cherche des calendriers valides, pas un calendrier optimal (on écrit $\min 0$).

In [1]:
import pulp

equipes = ["A", "B", "C", "D"]
journees = [1, 2, 3]

## Modèle

In [2]:
modele = pulp.LpProblem("phase_de_groupes", pulp.LpMinimize)

# x[i, j, k] = 1 si i joue contre j lors de la journée k
x = {(i, j, k): pulp.LpVariable(f"x_{i}_{j}_{k}", cat="Binary")
     for i in equipes for j in equipes if i != j for k in journees}

modele += 0   # pas d'objectif

# Symétrie
for (i, j, k) in x:
    modele += x[i, j, k] == x[j, i, k]

# Chaque paire se rencontre une fois
for i in equipes:
    for j in equipes:
        if i != j:
            modele += pulp.lpSum(x[i, j, k] for k in journees) == 1

# Un match par équipe et par journée
for i in equipes:
    for k in journees:
        modele += pulp.lpSum(x[i, j, k] for j in equipes if j != i) == 1

## Énumération de toutes les solutions

Après chaque calendrier trouvé, on ajoute une **coupe d'exclusion** qui l'interdit. Si $S$ est l'ensemble des matchs du calendrier (6 matchs, en ne gardant que $i < j$) :

$$\sum_{(i,j,k) \in S} x_{ijk} \le |S| - 1 = 5$$

Le calendrier déjà trouvé donne une somme de 6, il est donc exclu. Tout calendrier qui diffère d'au moins un match reste possible. Quand le solveur répond « infaisable », on a tout trouvé.

In [3]:
nb_solutions = 0
while True:
    modele.solve(pulp.PULP_CBC_CMD(msg=0))
    if pulp.LpStatus[modele.status] != "Optimal":
        break

    nb_solutions += 1
    S = [(i, j, k) for (i, j, k) in x if i < j and x[i, j, k].value() > 0.5]

    print(f"Calendrier {nb_solutions}")
    for k in journees:
        print(f"  Journée {k} : " + ", ".join(f"{i}-{j}" for (i, j, kk) in S if kk == k))

    modele += pulp.lpSum(x[v] for v in S) <= len(S) - 1

print(f"\nNombre total de calendriers : {nb_solutions}")

Calendrier 1
  Journée 1 : A-C, B-D
  Journée 2 : A-B, C-D
  Journée 3 : A-D, B-C
Calendrier 2
  Journée 1 : A-C, B-D
  Journée 2 : A-D, B-C
  Journée 3 : A-B, C-D
Calendrier 3
  Journée 1 : A-B, C-D
  Journée 2 : A-D, B-C
  Journée 3 : A-C, B-D
Calendrier 4
  Journée 1 : A-B, C-D
  Journée 2 : A-C, B-D
  Journée 3 : A-D, B-C
Calendrier 5
  Journée 1 : A-D, B-C
  Journée 2 : A-C, B-D
  Journée 3 : A-B, C-D
Calendrier 6
  Journée 1 : A-D, B-C
  Journée 2 : A-B, C-D
  Journée 3 : A-C, B-D

Nombre total de calendriers : 6


## Résultat

On obtient **6 calendriers**. C'est cohérent : il n'existe que 3 façons de former deux paires avec 4 équipes ({A-B, C-D}, {A-C, B-D}, {A-D, B-C}), et on peut les ordonner sur les 3 journées de $3! = 6$ façons.